In [1]:
import os
import glob
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, confusion_matrix, ConfusionMatrixDisplay, precision_score, recall_score, f1_score
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.svm import SVC
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv1D, MaxPooling1D, Flatten, Dense, Dropout
from tensorflow.keras.utils import to_categorical

MY_IP = "192.168.0.74"
MAX_LEN = 2000

In [2]:
#data path
dataF = np.load(r"/Users/nika/Desktop/Tor/raw/tor_200w_2500tr.npz", allow_pickle=True)
print(dataF.files)

['data', 'labels']


In [3]:
print("dataF keys:", dataF.files)

dataF keys: ['data', 'labels']


In [4]:
# Load arrays
X_F = dataF['data']
y_F = dataF['labels']

print("Total X shape:", X_F)
print("Total y shape:", y_F)

Total X shape: [[ 1  1 -1 ... -1 -1 -1]
 [ 1  1 -1 ... -1 -1 -1]
 [ 1  1 -1 ... -1 -1 -1]
 ...
 [ 1  1 -1 ... -1 -1 -1]
 [ 1  1 -1 ... -1 -1 -1]
 [ 1  1  1 ... -1 -1 -1]]
Total y shape: ['9gag.com' '9gag.com' '9gag.com' ... 'upornia.com' 'upornia.com'
 'upornia.com']


In [5]:
X_train, X_test, y_train, y_test = train_test_split(
    X_F,
    y_F,
    test_size=0.2,
    random_state=42,
    stratify=y_F
)
X_train_small=X_train.astype(np.float32)
X_test_small=X_test.astype(np.float32)
print("Train:", X_train_small.shape)
print("Test:", X_test_small.shape)

Train: (400000, 5000)
Test: (100000, 5000)


In [ ]:
scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

svm_npz1 = SVC(
    kernel="rbf",
    C=1,
    gamma="scale"
)

svm_npz1.fit(X_train_scaled, y_train)

y_pred_svm_npz1 = svm_npz1.predict(X_test_scaled)

acc_svm_npz1 = accuracy_score(y_test, y_pred_svm)

print("SVM Accuracy:", acc_svm_npz1)

In [ ]:
cm_svm_npz1 = confusion_matrix(y_test, y_pred_svm)

disp = ConfusionMatrixDisplay(
    confusion_matrix=cm_svm_npz1
)

disp.plot(xticks_rotation=90, cmap="Blues")
plt.savefig("confusion_matrix_svm.png", dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
# Precision, Recall, F1-score
precision = precision_score(y_test, y_pred_svm_npz1, average='weighted')
recall = recall_score(y_test, y_pred_svm_npz1, average='weighted')
f1 = f1_score(y_test, y_pred_svm_npz1, average='weighted')

print("SVMPrecision:", precision)
print("Recall:", recall)
print("F1-score:", f1)

In [ ]:
# ROC-кривая
svm_npz1_prob = SVC(
    kernel="rbf",
    C=1,
    gamma="scale",
    probability=True
)
svm_npz1_prob.fit(X_train_scaled, y_train)

# Получаем вероятности для класса 1
y_score = svm_npz1_prob.predict_proba(X_test_scaled)

# Если у тебя бинарная классификация
if len(np.unique(y_test)) == 2:
    fpr, tpr, thresholds = roc_curve(y_test, y_score[:, 1])
    roc_auc = auc(fpr, tpr)

    plt.figure()
    plt.plot(fpr, tpr, color='darkorange', lw=2, label='ROC curve (area = %0.2f)' % roc_auc)
    plt.plot([0, 1], [0, 1], color='navy', lw=2, linestyle='--')
    plt.xlim([0.0, 1.0])
    plt.ylim([0.0, 1.05])
    plt.xlabel('False Positive Rate')
    plt.ylabel('True Positive Rate')
    plt.title('SVM ROC Curve')
    plt.legend(loc="lower right")
    plt.show()
else:
    # Многоклассовая ROC
    y_test_bin = label_binarize(y_test, classes=np.unique(y_test))
    n_classes = y_test_bin.shape[1]

    plt.figure()
    for i in range(n_classes):
        fpr, tpr, _ = roc_curve(y_test_bin[:, i], y_score[:, i])
        roc_auc = auc(fpr, tpr)
        plt.plot(fpr, tpr, lw=2, label='Class %d (area = %0.2f)' % (i, roc_auc))
    
    plt.plot([0, 1], [0, 1], color='navy', lw=2, linestyle='--')
    plt.xlim([0.0, 1.0])
    plt.ylim([0.0, 1.05])
    plt.xlabel('False Positive Rate')
    plt.ylabel('True Positive Rate')
    plt.title('SVM ROC Curve (Multiclass)')
    plt.legend(loc="lower right")
    plt.show()